<a href="https://colab.research.google.com/github/falyseck/Multimodel_Data_Prepocessing/blob/main/data_merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd

social_url = "https://raw.githubusercontent.com/falyseck/Multimodel_Data_Prepocessing/main/data/raw/customer_info/customer_social_profiles.csv"
transactions_url = "https://raw.githubusercontent.com/falyseck/Multimodel_Data_Prepocessing/main/data/raw/customer_info/customer_transactions.csv"

social_df = pd.read_csv(social_url)
transactions_df = pd.read_csv(transactions_url)

# Quick check
print(social_df.head())
print(transactions_df.head())

  customer_id_new social_media_platform  engagement_score  \
0            A178              LinkedIn                74   
1            A190               Twitter                82   
2            A150              Facebook                96   
3            A162               Twitter                89   
4            A197               Twitter                92   

   purchase_interest_score review_sentiment  
0                      4.9         Positive  
1                      4.8          Neutral  
2                      1.6         Positive  
3                      2.6         Positive  
4                      2.3          Neutral  
   customer_id_legacy  transaction_id  purchase_amount purchase_date  \
0                 151            1001              408    2024-01-01   
1                 192            1002              332    2024-01-02   
2                 114            1003              442    2024-01-03   
3                 171            1004              256    2024-01-04 

In [13]:
print("Social Profiles:")
print(social_df.info())

print("\nTransactions:")
print(transactions_df.info())

Social Profiles:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customer_id_new          155 non-null    object 
 1   social_media_platform    155 non-null    object 
 2   engagement_score         155 non-null    int64  
 3   purchase_interest_score  155 non-null    float64
 4   review_sentiment         155 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 6.2+ KB
None

Transactions:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id_legacy  150 non-null    int64  
 1   transaction_id      150 non-null    int64  
 2   purchase_amount     150 non-null    int64  
 3   purchase_date       150 non-null    object 
 4   product_category

Since the IDs are not matching, we merge by row order

In [14]:
social_df = social_df.reset_index(drop=True)
transactions_df = transactions_df.reset_index(drop=True)

In [15]:
# Concatenate side by side
merged_df = pd.concat([social_df, transactions_df], axis=1)

# Drop the legacy ID column to avoid confusion
merged_df = merged_df.drop(columns=['customer_id_legacy'])

# Rename new ID column for consistency
merged_df = merged_df.rename(columns={'customer_id_new': 'customer_id'})

print("Merged Dataset Preview:")
display(merged_df.head())

Merged Dataset Preview:


,customer_id,social_media_platform,engagement_score,purchase_interest_score,review_sentiment,transaction_id,purchase_amount,purchase_date,product_category,customer_rating
0,A178,LinkedIn,74,4.9,Positive,1001.0,408.0,2024-01-01,Sports,2.3
1,A190,Twitter,82,4.8,Neutral,1002.0,332.0,2024-01-02,Electronics,4.2
2,A150,Facebook,96,1.6,Positive,1003.0,442.0,2024-01-03,Electronics,2.1
3,A162,Twitter,89,2.6,Positive,1004.0,256.0,2024-01-04,Clothing,2.8
4,A197,Twitter,92,2.3,Neutral,1005.0,64.0,2024-01-05,Clothing,1.3


In [16]:
merged_df.isnull().sum()

,0
customer_id,0
social_media_platform,0
engagement_score,0
purchase_interest_score,0
review_sentiment,0
transaction_id,5
purchase_amount,5
purchase_date,5
product_category,5
customer_rating,15


Handling missing values

In [18]:

# Fill customer_rating if missing
merged_df['customer_rating'] = merged_df['customer_rating'].fillna(merged_df['customer_rating'].mean())

# drop  remaining rows with NaNs
merged_df = merged_df.dropna()

In [19]:
merged_df.isnull().sum()

,0
customer_id,0
social_media_platform,0
engagement_score,0
purchase_interest_score,0
review_sentiment,0
transaction_id,0
purchase_amount,0
purchase_date,0
product_category,0
customer_rating,0


Encoding categorical columns

In [20]:
merged_df = pd.get_dummies(merged_df, columns=['social_media_platform', 'review_sentiment', 'product_category'])

Saving merged csv file

In [21]:
output_path = "merged_customer_data.csv"
merged_df.to_csv(output_path, index=False)
print("✅ Merged dataset saved to", output_path)
print("Shape:", merged_df.shape)

✅ Merged dataset saved to merged_customer_data.csv
Shape: (150, 20)
